# PyPSA to IFISC Model Data Converter

This notebook demonstrates how to extract data from a solved PyPSA network and convert it into the JSON format required by the IFISC `power_grid_modelV4.py`.

The process involves:
1.  Loading a solved PyPSA network.
2.  Extracting static data (nodes, lines, generators) and dynamic data (generator set-points, loads) for a single snapshot.
3.  Mapping and formatting this data into the specific nested JSON structure the IFISC model requires.
4.  Saving the final JSON file, which can then be used as an input to run a dynamic simulation with the IFISC model.

In [ ]:
import pypsa
import pandas as pd
import json
from datetime import datetime

# Load the solved network
# This network contains the results of an optimal power flow (OPF)
n = pypsa.Network("pypsa-eur/results/germany/networks/base_s_470_elec_.nc")

# Select a specific snapshot to analyze (e.g., the first hour)
snapshot = n.snapshots[0]
print(f"Using snapshot: {snapshot}")

## 1. Prepare the JSON Structure

First, we'll create the main dictionary that will hold all the data in the format required by the IFISC model. This involves setting up the default parameters and the nested dictionaries for static and dynamic data.

In [ ]:
# This dictionary will store all the data for the JSON file.
ifisc_data = {
    "defaults": {
        "time-step_(s)": 0.01,
        "reference_frequency_(Hz)": 50.0,
        "load_frequency_dependence": 1.0,
        "node_inertia_(MWs)": 1.0,
        "plants_primary_control_timescale_(s)": 10.0,
        "plants_set-point_timescale_when_on_(s)": 1.0,
        "plants_set-point_timescale_when_off_(s)": 1.0,
        "plants_governor_speed_regulator": 0.05,
        "plants_slow_switch_off_duration_(s)": 30.0,
        "minimum_line_impedance_(ohm)": 0.0001
    },
    "static_data": {
        "node_ID": [],
        "line_parameters": [],
        "external_line_parameters": [],
        "power_plant_parameters": [],
        "asset_parameters": []
    },
    "dynamic_data": []
}

## 2. Extract and Format Static Data

Here, we extract the static components of the grid from the PyPSA network.

-   **Nodes**: All buses in the PyPSA network become nodes in the IFISC model.
-   **Lines**: AC and DC lines from PyPSA are converted to the `line_parameters` format. We need to calculate the impedance `Z` from the PyPSA `x` (reactance) and `r` (resistance) values.
-   **Generators**: Conventional generators (like gas, coal) are mapped to `power_plant_parameters`. We need to provide values for inertia and control parameters, which are not always standard in PyPSA and may need to be assumed.
-   **Assets**: Renewable generators (like wind and solar) are treated as `asset_parameters` in the IFISC model. They are considered negative loads with a given generation profile.

In [ ]:
# Extract Node IDs
ifisc_data["static_data"]["node_ID"] = n.buses.index.tolist()

# Extract Line Parameters
for i, line in n.lines.iterrows():
    # Calculate impedance Z from resistance r and reactance x
    impedance = (line.r**2 + line.x**2)**0.5
    if impedance < ifisc_data["defaults"]["minimum_line_impedance_(ohm)"]:
        impedance = ifisc_data["defaults"]["minimum_line_impedance_(ohm)"]
        
    line_entry = {
        "initial_node": line.bus0,
        "final_node": line.bus1,
        "type": "AC",
        "nominal_voltage_(kV)": line.v_nom,
        "nominal_intensity_(kA)": line.s_nom / line.v_nom if line.v_nom > 0 else 0,
        "impedance_(ohm)": impedance
    }
    ifisc_data["static_data"]["line_parameters"].append(line_entry)

# DC lines are treated similarly but may need special handling if the IFISC model
# distinguishes them. Here we'll add them as standard lines.
for i, link in n.links.iterrows():
    # Assuming DC links are represented in n.links and have similar properties
    # This part might need adjustment based on how DC links are defined in your PyPSA model
    if link.p_nom > 0: # A simple check if it's a transmission link
        # PyPSA links don't have r and x, so we need to make an assumption or use typical values
        # For this example, we'll assume a default impedance.
        # A better approach would be to have this data in your PyPSA model.
        impedance = 0.01 
        line_entry = {
            "initial_node": link.bus0,
            "final_node": link.bus1,
            "type": "DC", # Or handle as a special type if IFISC model supports it
            "nominal_voltage_(kV)": n.buses.loc[link.bus0].v_nom, # Assuming same voltage
            "nominal_intensity_(kA)": link.p_nom / n.buses.loc[link.bus0].v_nom if n.buses.loc[link.bus0].v_nom > 0 else 0,
            "impedance_(ohm)": impedance
        }
        ifisc_data["static_data"]["line_parameters"].append(line_entry)


# Separate conventional generators from renewable "assets"
conventional_carriers = ['OCGT', 'CCGT', 'coal', 'lignite', 'nuclear', 'oil', 'geothermal']
conventional_gens = n.generators[n.generators.carrier.isin(conventional_carriers)]
renewable_gens = n.generators[~n.generators.carrier.isin(conventional_carriers)]

# Extract Power Plant Parameters (Conventional)
for i, gen in conventional_gens.iterrows():
    plant_entry = {
        "ID": gen.name,
        "node": gen.bus,
        "nominal_power_(MW)": gen.p_nom,
        # These are dynamic parameters not typically in PyPSA, so we use defaults/assumptions
        "inertia_(s)": 5.0, 
        "primary_control_capacity_(MW)": gen.p_nom * 0.1, # Assume 10% of p_nom
        "secondary_control_capacity_(MW/s)": 1.0
    }
    ifisc_data["static_data"]["power_plant_parameters"].append(plant_entry)

# Extract Asset Parameters (Renewables)
for i, gen in renewable_gens.iterrows():
    asset_entry = {
        "ID": gen.name,
        "node": gen.bus
    }
    ifisc_data["static_data"]["asset_parameters"].append(asset_entry)

## 3. Extract and Format Dynamic Data

This section prepares the time-dependent data for a single dispatch moment. The IFISC model can simulate over time from this starting point.

-   **Timestamp**: The date and time for the dispatch.
-   **Loads**: The active power demand at each node.
-   **Generator Set-points**: The optimal active power dispatch for each conventional generator, as calculated by PyPSA.
-   **Asset Generation**: The active power generation from renewable sources.

In [ ]:
# Create a dispatch entry for the selected snapshot
dispatch_entry = {
    "date_and_time": snapshot.isoformat(),
    "load_at_each_node": [],
    "power_plants_set_point": [],
    "asset_generation": []
}

# Extract Load Data
for i, load in n.loads_t.p_set.loc[snapshot].items():
    # The IFISC model expects positive values for load
    load_entry = {
        "node": n.loads.loc[i].bus,
        "active_power_(MW)": abs(load) 
    }
    dispatch_entry["load_at_each_node"].append(load_entry)

# Extract Power Plant Set-points (Conventional)
for gen_name in conventional_gens.index:
    set_point = n.generators_t.p.loc[snapshot, gen_name]
    plant_set_point_entry = {
        "ID": gen_name,
        "set_point_power_(MW)": set_point
    }
    dispatch_entry["power_plants_set_point"].append(plant_set_point_entry)

# Extract Asset Generation (Renewables)
for gen_name in renewable_gens.index:
    generation = n.generators_t.p.loc[snapshot, gen_name]
    asset_generation_entry = {
        "ID": gen_name,
        "active_power_(MW)": generation
    }
    dispatch_entry["asset_generation"].append(asset_generation_entry)

# Add the dispatch entry to the main data structure
ifisc_data["dynamic_data"].append(dispatch_entry)

## 4. Finalize and Save the JSON File

Finally, we'll add file paths for the output and save the complete dictionary as a JSON file. This file is now ready to be used with the `power_grid_modelV4.py` script.

In [ ]:
# Define output file paths
output_filename = "ifisc_input.json"
ifisc_data["files"] = {
    "output_data_file": "ifisc_output.json",
    "final_status_file": "ifisc_final_status.json",
    "continue_from_status_file": ""
}

# Save the dictionary to a JSON file
with open(output_filename, 'w') as f:
    json.dump(ifisc_data, f, indent=4)

print(f"Successfully created JSON file: {output_filename}")
print("\nThis file can now be used as input for the IFISC power grid model, for example:")
print(f"python power_grid_modelV4.py {output_filename}")